# Gold Daily Volatility Forecasting

This notebook uses a single forecasting model: a PyTorch neural network.

**Asset:** COMEX Gold Futures (`GC=F`)  
**Interval:** 1 day  
**Target:** next trading day's high-low range, normalized by the recent 20-day average range.

The goal is to forecast a volatility proxy, not the direction of the gold price.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Download daily gold data from Yahoo Finance

Yahoo Finance ticker `GC=F` represents COMEX Gold Futures. The download is saved as a CSV so the same historical dataset can be reused without downloading it every time.

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / "gold_daily.csv"

if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH, parse_dates=["date"], index_col="date")
    print("Loaded existing CSV:", CSV_PATH)
else:
    raw = yf.download(
        "GC=F",
        start="2000-01-01",
        interval="1d",
        auto_adjust=False,
        progress=False
    )

    if raw.empty:
        raise RuntimeError("Yahoo Finance returned no data. Check your internet connection or ticker.")

    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    raw.columns = [str(c).lower().replace(" ", "_") for c in raw.columns]
    raw.index.name = "date"
    df = raw.copy()
    df.to_csv(CSV_PATH)
    print("Downloaded and saved:", CSV_PATH)

print("Rows:", len(df))
print("First date:", df.index.min())
print("Last date:", df.index.max())
display(df.head())

## 2. Clean the data

In [ ]:
df = df.copy()
df.index = pd.to_datetime(df.index)
df = df[~df.index.duplicated(keep="first")].sort_index()

required = ["open", "high", "low", "close", "volume"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required Yahoo Finance columns: {missing}")

for col in required:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["open", "high", "low", "close"])
df = df[(df["open"] > 0) & (df["high"] > 0) & (df["low"] > 0) & (df["close"] > 0)]

print("Clean rows:", len(df))
display(df[required].describe())

## 3. Feature engineering

Only past and current-day information is used to create features. The target is shifted one trading day into the future.

In [ ]:
df["intraday_return"] = (df["close"] - df["open"]) / df["open"] * 100
df["daily_range"] = (df["high"] - df["low"]) / df["low"] * 100
df["close_return"] = np.log(df["close"] / df["close"].shift(1)) * 100

# Compact volatility and range features
for window in [5, 20]:
    df[f"volatility_{window}"] = df["close_return"].rolling(window).std()

df["range_ma_20"] = df["daily_range"].rolling(20).mean()

# Short-term lag features
df["return_lag_1"] = df["close_return"].shift(1)
df["range_lag_1"] = df["daily_range"].shift(1)
df["range_lag_2"] = df["daily_range"].shift(2)

# Trend and volatility regime
df["ma_20"] = df["close"].rolling(20).mean()
df["distance_ma_20"] = (df["close"] - df["ma_20"]) / (df["ma_20"] + 1e-8)
df["volatility_regime"] = df["volatility_5"] / (df["volatility_20"] + 1e-8)
df["rolling_high_60"] = df["close"].rolling(60).max()
df["drawdown_60"] = (df["close"] - df["rolling_high_60"]) / (df["rolling_high_60"] + 1e-8)

# Predict next trading day range relative to the recent average range
df["next_day_range"] = df["daily_range"].shift(-1)
df["target"] = df["next_day_range"] / (df["range_ma_20"] + 1e-8)

display(df.tail())


## 4. Select features and prepare the dataset

In [ ]:
feature_cols = [
    "close_return",
    "intraday_return",
    "daily_range",
    "volatility_5",
    "volatility_20",
    "range_ma_20",
    "range_lag_1",
    "range_lag_2",
    "return_lag_1",
    "distance_ma_20",
    "volatility_regime",
    "drawdown_60"
]

needed = feature_cols + ["target"]
df_clean = df.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna(subset=needed).copy()

print("Total usable rows:", len(df_clean))
print("Number of features:", len(feature_cols))
print("Selected features:", feature_cols)
print("First usable date:", df_clean.index.min())
print("Last usable date:", df_clean.index.max())

if len(df_clean) < 500:
    raise ValueError("Not enough historical rows to train and evaluate the models.")


## 5. Chronological train/test split

The last 20% of observations are kept for testing. The scaler is fitted only on the training data to avoid leaking information from the test period.

In [ ]:
split_point = int(len(df_clean) * 0.8)

X = df_clean[feature_cols].values.astype("float32")
y = df_clean["target"].values.astype("float32")

X_train_raw, X_test_raw = X[:split_point], X[split_point:]
y_train, y_test = y[:split_point], y[split_point:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype("float32")
X_test = scaler.transform(X_test_raw).astype("float32")

train_dates = df_clean.index[:split_point]
test_dates = df_clean.index[split_point:]

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("Training period:", train_dates.min(), "to", train_dates.max())
print("Testing period: ", test_dates.min(), "to", test_dates.max())

## 6. PyTorch neural network

In [ ]:
model = nn.Sequential(
    nn.Linear(len(feature_cols), 32),
    nn.ReLU(),
    nn.Linear(32, 1),
    nn.Softplus()
).to(DEVICE)

print(model)

X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

loader = DataLoader(
    TensorDataset(X_tensor, y_tensor),
    batch_size=128,
    shuffle=True
)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 30
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        predictions = model(xb)
        loss = loss_fn(predictions, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:02d}/{EPOCHS} | Loss: {total_loss / len(loader):.6f}")

## 7. Evaluate the neural network

Only the neural network is trained and evaluated in this notebook.

In [ ]:
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
    nn_predictions = model(X_test_tensor).cpu().numpy().ravel()

def evaluate_model(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted)
    }

results_df = pd.DataFrame([evaluate_model(y_test, nn_predictions)], index=["Neural Network"])
display(results_df)


## 8. Actual vs predicted volatility

Compare the neural network forecast with the observed next-day volatility proxy.

In [ ]:
comparison = pd.DataFrame({
    "actual": y_test,
    "neural_network": nn_predictions
}, index=test_dates)

n = min(250, len(comparison))
plt.figure(figsize=(14, 5))
plt.plot(comparison.index[:n], comparison["actual"].iloc[:n], label="Actual", linewidth=1.5)
plt.plot(comparison.index[:n], comparison["neural_network"].iloc[:n], label="Neural Network", alpha=0.85)
plt.title("Gold Next-Day Normalized Range — Test Set")
plt.xlabel("Date")
plt.ylabel("Normalized range")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(y_test, nn_predictions, alpha=0.35, s=15)
low = min(y_test.min(), nn_predictions.min())
high = max(y_test.max(), nn_predictions.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.title("Neural Network — Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 9. Save test predictions

In [ ]:
OUTPUT_PATH = DATA_DIR / "gold_test_predictions.csv"
comparison.to_csv(OUTPUT_PATH)
print("Saved:", OUTPUT_PATH)
display(comparison.tail())


## Notes and limitations

- The target is a next-day **volatility proxy** (normalized daily high-low range), not tomorrow's closing price or market direction.
- `GC=F` is a gold futures ticker. Yahoo Finance history, volume values and symbol coverage may change.
- The first version uses OHLCV-derived features only. It does not yet include news, sentiment, interest rates or the US dollar index.
- Model scores on one historical test period do not prove that the model will work in the future. A later version should use walk-forward validation, compare against rolling baselines, and account for market regime changes.
- This notebook is for research and education, not financial advice.